In [1]:
import sys
sys.path.insert(1, '/workspaces/aml-working-paper/aml_working_paper/routing')

In [2]:
import sys
sys.path.insert(1, '/workspaces/aml-working-paper/aml_working_paper/data')

In [3]:
from routing_investigate import build_candidate_windows, select_investigations, validate_investigations
from local_sample import prepare_local_windows
from ground_truth import get_ground_truth
import pandas as pd
from preprocess import preprocess_data

In [4]:
routing_df = pd.read_parquet("/workspaces/aml-working-paper/aml_working_paper/data/routing/routing_df.parquet")

In [5]:
candidates = pd.read_parquet("//workspaces/aml-working-paper/aml_working_paper/data/routing/candidates.parquet")

In [6]:
df_aml = pd.read_csv("/workspaces/aml-working-paper/aml_working_paper/data/synthetic_df_aml.csv")

In [7]:
train_df, val_df, test_df, train_sequences, val_sequences, test_sequences, scaler = preprocess_data(df_aml)

In [8]:
features = [
    "economic_context",
    "network_exposure",
    "structuring_score",
    "cross_border_ratio",
    "transaction_volume",
]

selected = select_investigations(
    candidates,
    budget=10,
    strategy="uncertainty",
)

local_windows = prepare_local_windows(
    data=val_df,
    selected=selected,
    features=features,
)

print("Number of windows:", len(local_windows))
print(local_windows[0]["data"])


Number of windows: 10
    time  economic_context  network_exposure  structuring_score  \
18    18         -0.985434          0.061948           0.600674   
19    19         -0.712667          0.074647           0.653479   
20    20         -0.030768          0.205329           0.337485   
21    21         -0.709335          0.080911           0.574244   
22    22         -0.344557          0.047874           0.315098   
23    23          0.358036          0.141916           0.765080   
24    24          0.699606          0.081910           0.531208   
25    25          0.790572          0.143995           0.486489   
26    26          0.100495          0.102052           0.900002   
27    27         -0.292715          0.271902           0.145264   

    cross_border_ratio  transaction_volume  
18            0.311376           23.353993  
19            0.340143           29.255242  
20            0.255258           13.256029  
21            0.112332           13.762872  
22            0

In [9]:

for window in local_windows:
    print(
        window["account_id"],
        window["start_time"],
        window["end_time"],
        len(window["data"]),
    )


55.0 18.0 27.0 10
70.0 1.0 10.0 10
12.0 70.0 79.0 10
53.0 83.0 92.0 10
33.0 74.0 83.0 10
33.0 13.0 22.0 10
30.0 81.0 90.0 10
40.0 0.0 9.0 10
15.0 88.0 97.0 10
44.0 62.0 71.0 10


In [10]:
print(local_windows[0]["data"].columns.tolist())

['time', 'economic_context', 'network_exposure', 'structuring_score', 'cross_border_ratio', 'transaction_volume']


In [11]:
current_edges, temporal_edges = get_ground_truth()

print("Current edges:", len(current_edges))
print("Temporal edges:", len(temporal_edges))

Current edges: 8
Temporal edges: 2


In [12]:
from aml_working_paper.causal.discovery_data import (
    prepare_discovery_data,
)

discovery_data = prepare_discovery_data(local_windows)

print(discovery_data.shape)
print(discovery_data.head())

(100, 8)
   time  economic_context  network_exposure  structuring_score  \
0    18         -0.985434          0.061948           0.600674   
1    19         -0.712667          0.074647           0.653479   
2    20         -0.030768          0.205329           0.337485   
3    21         -0.709335          0.080911           0.574244   
4    22         -0.344557          0.047874           0.315098   

   cross_border_ratio  transaction_volume  window_id  account_id  
0            0.311376           23.353993          0          55  
1            0.340143           29.255242          0          55  
2            0.255258           13.256029          0          55  
3            0.112332           13.762872          0          55  
4            0.187702           18.888078          0          55  


In [13]:
features = [
    "economic_context",
    "network_exposure",
    "structuring_score",
    "cross_border_ratio",
    "transaction_volume",
]

# Check missing values
print(discovery_data[features].isna().sum())

# Check variability
print(discovery_data[features].nunique())

# Check observations per window
print(discovery_data.groupby("window_id").size())

economic_context      0
network_exposure      0
structuring_score     0
cross_border_ratio    0
transaction_volume    0
dtype: int64
economic_context       62
network_exposure      100
structuring_score     100
cross_border_ratio    100
transaction_volume    100
dtype: int64
window_id
0    10
1    10
2    10
3    10
4    10
5    10
6    10
7    10
8    10
9    10
dtype: int64
